In [ ]:
# NOTE: 03_B_DeepDisc.ipynb - Cell 1
# Run with the deepdisc_IG kernel.
# See 02_C_DeepDisc_Algorithm_Test.ipynb Cell 1 for the install recipe.
#
# This cell does everything end-to-end:
#   1. Discovers all star folders under BASE_DATA_DIR
#   2. Builds a COCO training dataset from DAOStarFinder detections
#   3. Trains a Faster R-CNN (ResNet-50-FPN) model via detectron2
#   4. Runs the trained model on every plate for every star
#   5. Applies the APASS-calibrated aperture photometry pipeline from 02_A
#   6. Saves lightcurve.csv and photometry_results.csv per star
#
# All intermediate results are cached -- re-runs only process new/changed plates.
# Delete _shared_cache_03_B/ to force a full rebuild from scratch.
#
# ── CONFIG ──────────────────────────────────────────────────────────────────
# Set DEBUG_RUN = True for a 50-iteration smoke test. Confirm it completes
# without errors, then set False and re-run for a full training run.
DEBUG_RUN = True

# ── IMPORTS ──────────────────────────────────────────────────────────────────
import sys, shelve, time, json, warnings, traceback, os, copy, random
import urllib.request, urllib.parse
import cv2, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
from io import BytesIO
from PIL import Image as PILImage, Image
from IPython.display import display, clear_output
from tqdm import tqdm

from astropy.io import fits as astrofits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astropy.stats import SigmaClip, sigma_clip
import astropy.units as u
from photutils.aperture import (CircularAperture, CircularAnnulus,
                                 ApertureStats, aperture_photometry)
from photutils.centroids import centroid_sources, centroid_com
from photutils.background import Background2D, MedianBackground

import detectron2.data.transforms as T
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.modeling import build_model
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator
from detectron2.data import (DatasetCatalog, MetadataCatalog,
                              build_detection_train_loader,
                              build_detection_test_loader,
                              detection_utils as utils)
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.logger import setup_logger
from deepdisc.data_format.image_readers import ImageReader

warnings.filterwarnings('ignore')
setup_logger()

SCRIPTS_DIR = Path(r'C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\scripts')
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
for _mod in list(sys.modules.keys()):
    if 'plate_scan_utils' in _mod:
        del sys.modules[_mod]
import importlib
from plate_scan_utils import detect_sources, detect_plate_errors

# ── PATHS ─────────────────────────────────────────────────────────────────────
BASE_DATA_DIR = Path(r'C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data')
SHARED_CACHE  = BASE_DATA_DIR / '_shared_cache_03_B'
SHARED_CACHE.mkdir(parents=True, exist_ok=True)

COCO_DIR    = SHARED_CACHE / 'coco_dataset'
IMG_DIR     = COCO_DIR / 'images'
MODEL_DIR   = SHARED_CACHE / 'deepdisc_model'
COCO_JSON   = COCO_DIR / 'annotations.json'
BUILD_CACHE = COCO_DIR / 'build_cache.json'
for d in (COCO_DIR, IMG_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── STAR DISCOVERY ─────────────────────────────────────────────────────────────
star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir()
                     if d.is_dir() and (d / 'cutouts').is_dir()
                     and not d.name.startswith('_')])
print(f'Found {len(star_dirs)} star folder(s):')
for d in star_dirs:
    print(f'  {d.name}: {len(list((d/"cutouts").glob("*.fits")))} plates')

# ── APASS CACHE ────────────────────────────────────────────────────────────────
apass_cache = shelve.open(str(SHARED_CACHE / 'apass_cache_shelf'), flag='c', writeback=False)
print(f'\nAPASS cache: {len(apass_cache)} field queries')

# ── COORDINATE RESOLUTION ──────────────────────────────────────────────────────
MANUAL_TARGET_COORDS = {
    'V_CrA': SkyCoord(ra=281.884623417 * u.deg, dec=-38.158974417 * u.deg),
}
TARGET_COORD_CACHE = SHARED_CACHE / 'target_coords.csv'

def resolve_target_coord(star_dir):
    name = star_dir.name
    if name in MANUAL_TARGET_COORDS:
        return MANUAL_TARGET_COORDS[name], 'manual'
    if TARGET_COORD_CACHE.exists():
        df = pd.read_csv(TARGET_COORD_CACHE)
        m = df[df['folder_name'] == name]
        if len(m) > 0:
            r = m.iloc[0]
            return SkyCoord(ra=r['ra_deg'] * u.deg, dec=r['dec_deg'] * u.deg), 'cached'
    try:
        coord = SkyCoord.from_name(name.replace('_', ' '))
        row = pd.DataFrame([{'folder_name': name, 'query_name': name.replace('_', ' '),
                              'ra_deg': coord.ra.deg, 'dec_deg': coord.dec.deg}])
        if TARGET_COORD_CACHE.exists():
            row.to_csv(TARGET_COORD_CACHE, mode='a', header=False, index=False)
        else:
            row.to_csv(TARGET_COORD_CACHE, index=False)
        return coord, 'resolved'
    except Exception as e:
        return None, f'FAILED: {e}'

target_coords = {}
for star_dir in star_dirs:
    coord, src = resolve_target_coord(star_dir)
    target_coords[star_dir.name] = (coord, src)
    print(f'  {star_dir.name}: {"OK" if coord is not None else "FAILED"} ({src})')

# ── PHOTOMETRY PARAMETERS (identical to 02_A) ──────────────────────────────────
DEFAULT_APERTURE      = 6
ANNULUS_INNER         = 15
ANNULUS_OUTER         = 20
CENTROID_BOX          = 21
CENTROID_MAX_DRIFT    = 5
SIGNIF_THRESHOLD      = 5.0
SATURATION_FRACTION   = 0.99
SATURATION_MIN_PIXELS = 5
ISOLATION_MIN_SEP     = ANNULUS_OUTER + 2
OUTLIER_SIGMA         = 3.0
MIN_REF_STARS         = 25
LOCAL_HALF_WINDOW     = 1.5
MIN_REF_STARS_LOCAL   = 8
LOCAL_WINDOW_GROWTH   = 1.6
MAX_CALIBRATION_RMS   = 0.5
MIN_APASS_STARS_QUERY = 10

# ── PHOTOMETRY FUNCTIONS (verbatim from 02_A) ──────────────────────────────────
def subtract_background_2d(data, box_size=50):
    try:
        bkg = Background2D(data, (box_size, box_size), filter_size=(3, 3),
                            bkg_estimator=MedianBackground())
        return data - bkg.background, bkg.background_rms
    except Exception:
        med = np.nanmedian(data)
        return data - med, np.full_like(data, np.nanstd(data))

def refine_centroids(data, x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if len(x) == 0:
        return x, y
    try:
        xr, yr = centroid_sources(data, x, y, box_size=CENTROID_BOX, centroid_func=centroid_com)
        good = np.isfinite(xr) & np.isfinite(yr) & (np.hypot(xr - x, yr - y) < CENTROID_MAX_DRIFT)
        xr[~good], yr[~good] = x[~good], y[~good]
        return xr, yr
    except Exception:
        return x, y

def filter_isolated(x, y):
    n = len(x)
    if n < 2:
        return np.ones(n, dtype=bool)
    iso = np.ones(n, dtype=bool)
    for i in range(n):
        d = np.hypot(x - x[i], y - y[i]); d[i] = np.inf
        if np.any(d < ISOLATION_MIN_SEP):
            iso[i] = False
    return iso

def measure_photometry(data, x, y):
    if len(x) == 0:
        return np.array([]), np.array([]), np.array([])
    pos = list(zip(x, y))
    ap  = CircularAperture(pos, r=DEFAULT_APERTURE)
    ann = CircularAnnulus(pos, r_in=ANNULUS_INNER, r_out=ANNULUS_OUTER)
    st  = ApertureStats(data, ann, sigma_clip=SigmaClip(sigma=3.0))
    bm  = np.nan_to_num(st.median, nan=0.0)
    bs  = np.nan_to_num(st.std,    nan=0.0)
    ph  = aperture_photometry(data, ap)
    fl  = ph['aperture_sum'].value - bm * (np.pi * DEFAULT_APERTURE**2)
    return fl, bm, bs

def detect_saturation(data, x, y):
    n = len(x); sat = np.zeros(n, dtype=bool)
    dm = np.nanmax(data)
    if dm <= 0:
        return sat
    masks = CircularAperture(list(zip(x, y)), r=DEFAULT_APERTURE).to_mask(method='center')
    for i, mask in enumerate(masks):
        cut = mask.multiply(data)
        if cut is None:
            continue
        px = cut[mask.data > 0]
        sat[i] = np.sum(px >= SATURATION_FRACTION * dm) >= SATURATION_MIN_PIXELS
    return sat

def calibrate_global(inst, cat):
    mask = np.isfinite(inst) & np.isfinite(cat)
    if mask.sum() < MIN_REF_STARS:
        return None
    x, y = cat[mask], inst[mask]
    for _ in range(3):
        try:
            c = np.polyfit(x, y, 2); r = y - np.polyval(c, x)
            rms = np.sqrt(np.mean(r**2)); good = np.abs(r) < OUTLIER_SIGMA * rms
            if good.sum() < MIN_REF_STARS:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < MIN_REF_STARS:
        return None
    c = np.polyfit(x, y, 2); r = y - np.polyval(c, x)
    return {'coeffs': c, 'rms': float(np.sqrt(np.mean(r**2))),
            'n_used': len(x), 'b_min': float(x.min()), 'b_max': float(x.max())}

def calibrate_local(inst, cat, b_est):
    mask = np.isfinite(inst) & np.isfinite(cat)
    xa, ya = cat[mask], inst[mask]
    if len(xa) < MIN_REF_STARS_LOCAL:
        return None
    win = LOCAL_HALF_WINDOW
    lm  = np.abs(xa - b_est) <= win
    for _ in range(5):
        if lm.sum() >= MIN_REF_STARS_LOCAL:
            break
        win *= LOCAL_WINDOW_GROWTH; lm = np.abs(xa - b_est) <= win
    if lm.sum() < MIN_REF_STARS_LOCAL:
        return None
    x, y = xa[lm], ya[lm]
    for _ in range(3):
        try:
            c = np.polyfit(x, y, 1); r = y - np.polyval(c, x)
            rms = np.sqrt(np.mean(r**2)); good = np.abs(r) < OUTLIER_SIGMA * rms
            if good.sum() < MIN_REF_STARS_LOCAL:
                break
            x, y = x[good], y[good]
        except Exception:
            return None
    if len(x) < MIN_REF_STARS_LOCAL:
        return None
    c = np.polyfit(x, y, 1); r = y - np.polyval(c, x)
    return {'coeffs': c, 'rms': float(np.sqrt(np.mean(r**2))), 'n_used': len(x),
            'b_min': float(x.min()), 'b_max': float(x.max())}

def invert_calibration(cal, t_inst):
    c = list(cal['coeffs']); c[-1] -= t_inst
    roots = np.roots(c)
    real  = roots[np.abs(roots.imag) < 1e-6].real
    if len(real) == 0:
        return None
    lo, hi = cal['b_min'], cal['b_max']
    inr = real[(real >= lo - 2) & (real <= hi + 2)]
    cand = inr if len(inr) > 0 else real
    return float(min(cand, key=lambda r: abs(np.polyval(cal['coeffs'], r) - t_inst)))

def query_apass(ra_c, dec_c, radius_arcsec):
    params = {'-source': 'II/336/apass9', '-c': f'{ra_c} {dec_c}',
              '-c.rs': radius_arcsec, '-out': 'RAJ2000,DEJ2000,Bmag,e_Bmag', '-out.max': 3000}
    url = 'https://vizier.cds.unistra.fr/viz-bin/asu-tsv?' + urllib.parse.urlencode(params)
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode('utf-8')
        except Exception:
            time.sleep(2 ** attempt)
    return None

def parse_apass_tsv(text):
    rows = []
    for line in text.split('\n'):
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('-'):
            continue
        parts = line.split('\t')
        if len(parts) < 4:
            continue
        try:
            ra, dec, bm, eb = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
            rows.append((ra, dec, bm, eb))
        except Exception:
            continue
    if rows and not (0 < rows[0][2] < 25):
        rows = rows[1:]
    return rows

def get_field_catalog(ra_c, dec_c, radius_arcsec):
    key = f'{round(ra_c, 3)}_{round(dec_c, 3)}'
    if key in apass_cache:
        return apass_cache[key]
    raw = query_apass(ra_c, dec_c, radius_arcsec)
    rows = [r for r in parse_apass_tsv(raw) if 0 < r[2] < 20 and r[3] < 0.5] if raw else []
    apass_cache[key] = rows
    return rows

def get_plate_jd(header):
    if 'JD-OBS'  in header: return float(header['JD-OBS'])
    if 'MJD-OBS' in header: return float(header['MJD-OBS']) + 2400000.5
    for k in ('DATE-OBS', 'DATE'):
        if k in header:
            try: return Time(header[k]).jd
            except Exception: continue
    return np.nan

def fits_to_png(fits_path, png_path):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    lo, hi = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    norm = np.clip((data - lo) / (hi - lo + 1e-9), 0, 1)
    Image.fromarray((norm * 255).astype(np.uint8)).convert('RGB').save(png_path)
    return data.shape

def format_eta(s):
    if s is None or s != s or s < 0: return 'calculating...'
    m, s = divmod(int(s), 60); h, m = divmod(m, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m}:{s:02d}'

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1: BUILD COCO DATASET
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('SECTION 1: Building COCO dataset')
print('='*60)

CATEGORIES = [
    {'id': 0, 'name': 'star'},    {'id': 1, 'name': 'target'},
    {'id': 2, 'name': 'scratch'}, {'id': 3, 'name': 'trailing'},
    {'id': 4, 'name': 'saturation'}, {'id': 5, 'name': 'dust'},
]
BOX_HALF = 6.0
LINE_PAD = 4.0

if COCO_JSON.exists() and BUILD_CACHE.exists():
    with open(COCO_JSON) as f: existing_coco = json.load(f)
    with open(BUILD_CACHE) as f: build_cache = json.load(f)
    images      = existing_coco['images']
    annotations = existing_coco['annotations']
    img_id  = max((i['id'] for i in images),      default=-1) + 1
    ann_id  = max((a['id'] for a in annotations), default=-1) + 1
    print(f'Loaded existing COCO: {len(images)} images, {len(annotations)} annotations')
else:
    images, annotations, build_cache = [], [], {}
    img_id = ann_id = 0
    print('No existing cache -- full build')

existing_stems = {Path(img['file_name']).stem for img in images}

for star_dir in star_dirs:
    target_coord, _ = target_coords[star_dir.name]
    cutouts = sorted((star_dir / 'cutouts').glob('*.fits'))
    cache_dir = star_dir / '03_B' / 'coco_cache'
    cache_dir.mkdir(parents=True, exist_ok=True)
    plate_cache = shelve.open(str(cache_dir / 'plate_shelf'), flag='c', writeback=False)
    sn = sc = ss = 0

    for fits_path in tqdm(cutouts, desc=star_dir.name, leave=False):
        mtime    = fits_path.stat().st_mtime
        png_path = IMG_DIR / (fits_path.stem + '.png')
        c_mtime  = build_cache.get(str(fits_path))

        if (c_mtime is not None and abs(c_mtime - mtime) < 1.0
                and png_path.exists() and fits_path.stem in existing_stems):
            sc += 1; continue

        try:
            sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(fits_path)
            errors = detect_plate_errors(data)
            h, w   = fits_to_png(fits_path, png_path)
        except Exception:
            ss += 1; continue

        if sources is None or len(sources) == 0:
            ss += 1; continue

        xs = np.array(sources[x_col], float)
        ys = np.array(sources[y_col], float)

        target_idx = -1
        if target_coord is not None:
            try:
                wcs = WCS(astrofits.getheader(fits_path))
                tx, ty = wcs.all_world2pix(target_coord.ra.deg, target_coord.dec.deg, 0)
                dists  = np.hypot(xs - float(tx), ys - float(ty))
                if dists.min() < 15:
                    target_idx = int(np.argmin(dists))
            except Exception:
                pass

        images.append({'id': img_id, 'file_name': str(png_path.resolve()), 'height': h, 'width': w})
        existing_stems.add(fits_path.stem)

        for i, (x, y) in enumerate(zip(xs, ys)):
            cat = 1 if i == target_idx else 0
            bx0, by0 = x - BOX_HALF, y - BOX_HALF; bw = bh = 2 * BOX_HALF
            annotations.append({'id': ann_id, 'image_id': img_id, 'category_id': cat,
                                 'bbox': [float(bx0), float(by0), float(bw), float(bh)],
                                 'area': float(bw * bh), 'iscrowd': 0})
            ann_id += 1

        for s in errors.get('scratches', []):
            x0, y0, x1, y1 = s['x0'], s['y0'], s['x1'], s['y1']
            bx0, by0 = min(x0, x1) - LINE_PAD, min(y0, y1) - LINE_PAD
            bw = abs(x1 - x0) + 2 * LINE_PAD; bh = abs(y1 - y0) + 2 * LINE_PAD
            annotations.append({'id': ann_id, 'image_id': img_id, 'category_id': 2,
                                 'bbox': [float(bx0), float(by0), float(bw), float(bh)],
                                 'area': float(bw * bh), 'iscrowd': 0})
            ann_id += 1
        for t in errors.get('trailing', []):
            bw, bh = t['w'], t['h']; bx0, by0 = t['x'] - bw/2, t['y'] - bh/2
            annotations.append({'id': ann_id, 'image_id': img_id, 'category_id': 3,
                                 'bbox': [float(bx0), float(by0), float(bw), float(bh)],
                                 'area': float(bw * bh), 'iscrowd': 0})
            ann_id += 1
        for cat_id, key in [(4, 'saturation'), (5, 'dust')]:
            for c in errors.get(key, []):
                bw = bh = 2 * c['r']; bx0, by0 = c['x'] - c['r'], c['y'] - c['r']
                annotations.append({'id': ann_id, 'image_id': img_id, 'category_id': cat_id,
                                     'bbox': [float(bx0), float(by0), float(bw), float(bh)],
                                     'area': float(bw * bh), 'iscrowd': 0})
                ann_id += 1

        plate_cache[str(fits_path)] = {'errors': errors, 'n_sources': len(xs), '_ver': 1}
        build_cache[str(fits_path)] = mtime
        img_id += 1; sn += 1

    plate_cache.sync(); plate_cache.close()
    print(f'  {star_dir.name}: {sn} new | {sc} cached | {ss} skipped')

with open(COCO_JSON, 'w') as f:
    json.dump({'images': images, 'annotations': annotations, 'categories': CATEGORIES}, f)
with open(BUILD_CACHE, 'w') as f:
    json.dump(build_cache, f)
print(f'COCO done: {len(images)} images, {len(annotations)} annotations')

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2: TRAIN/VAL SPLIT (auto-select paradigm plate)
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('SECTION 2: Train/val split')
print('='*60)

stem_to_coco_img = {Path(img['file_name']).stem: img for img in images}
imgid_to_anns    = {}
for ann in annotations:
    imgid_to_anns.setdefault(ann['image_id'], []).append(ann)

TRAIN_JSON = COCO_DIR / 'train_annotations.json'
VAL_JSON   = COCO_DIR / 'val_annotations.json'

# Auto-select paradigm plate: most star annotations, fewest error annotations
best_score, paradigm_imgid = -1, None
for img in images:
    anns    = imgid_to_anns.get(img['id'], [])
    n_stars = sum(1 for a in anns if a['category_id'] in (0, 1))
    n_errs  = sum(1 for a in anns if a['category_id'] > 1)
    score   = n_stars - 3 * n_errs
    if score > best_score:
        best_score = score; paradigm_imgid = img['id']

random.seed(42)
non_par    = [img for img in images if img['id'] != paradigm_imgid]
random.shuffle(non_par)
split      = int(0.8 * len(non_par))
par_entry  = next(img for img in images if img['id'] == paradigm_imgid)
train_imgs = [par_entry] + non_par[:split]
val_imgs   = non_par[split:]
train_ids  = {img['id'] for img in train_imgs}
val_ids    = {img['id'] for img in val_imgs}

for path, imgs, ids in [(TRAIN_JSON, train_imgs, train_ids), (VAL_JSON, val_imgs, val_ids)]:
    anns = [a for a in annotations if a['image_id'] in ids]
    with open(path, 'w') as f:
        json.dump({'images': imgs, 'annotations': anns, 'categories': CATEGORIES}, f)
    print(f'Wrote {path.name}  ({len(imgs)} images, {len(anns)} annotations)')

for name in ('plates_train', 'plates_val'):
    if name in DatasetCatalog: DatasetCatalog.remove(name)
    if name in MetadataCatalog: MetadataCatalog.remove(name)
register_coco_instances('plates_train', {}, str(TRAIN_JSON), str(IMG_DIR))
register_coco_instances('plates_val',   {}, str(VAL_JSON),   str(IMG_DIR))
cat_names = [c['name'] for c in CATEGORIES]
MetadataCatalog.get('plates_train').thing_classes = cat_names
MetadataCatalog.get('plates_val').thing_classes   = cat_names
print(f'Paradigm plate: image_id={paradigm_imgid}  |  score={best_score}')
print(f'Train: {len(train_imgs)}  |  Val: {len(val_imgs)}')

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3: MODEL CONFIG
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print(f'SECTION 3: Model config  (DEBUG_RUN={DEBUG_RUN})')
print('='*60)

class PNGImageReader(ImageReader):
    # Reads normalized RGB PNGs. Detectron2 expects BGR uint8.
    def _read_image(self, file_path, *args, **kwargs):
        img = cv2.imread(str(file_path))
        if img is None:
            img_rgb = np.array(PILImage.open(file_path).convert('RGB'))
            img = img_rgb[:, :, ::-1].copy()
        return img

imreader = PNGImageReader(norm='raw')

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file('COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml'))
cfg.DATASETS.TRAIN = ('plates_train',)
cfg.DATASETS.TEST  = ('plates_val',)
cfg.MODEL.WEIGHTS  = model_zoo.get_checkpoint_url('COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml')
cfg.MODEL.DEVICE   = 'cpu'
cfg.DATALOADER.NUM_WORKERS       = 0  # Must be 0 on Windows
cfg.INPUT.MIN_SIZE_TRAIN         = (512,)
cfg.INPUT.MAX_SIZE_TRAIN         = 1024
cfg.INPUT.MIN_SIZE_TEST          = 512
cfg.INPUT.MAX_SIZE_TEST          = 1024
cfg.SOLVER.IMS_PER_BATCH         = 1
cfg.SOLVER.BASE_LR               = 0.00025
cfg.SOLVER.MAX_ITER              = 50   if DEBUG_RUN else 2000
cfg.SOLVER.STEPS                 = ()
cfg.SOLVER.CHECKPOINT_PERIOD     = 25   if DEBUG_RUN else 500
cfg.SOLVER.WARMUP_ITERS          = 0
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 64
cfg.MODEL.ROI_HEADS.NUM_CLASSES          = 6
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST    = 0.5
# Small anchors (8,16px) for point sources; larger for scratches/saturation/dust.
cfg.MODEL.ANCHOR_GENERATOR.SIZES         = [[8, 16, 32, 64, 128]]
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.5, 1.0, 2.0]]
cfg.OUTPUT_DIR = str(MODEL_DIR)
print(f'MAX_ITER={cfg.SOLVER.MAX_ITER}  |  output: {MODEL_DIR}')

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4: TRAINING
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('SECTION 4: Training')
print('='*60)

class PNGMapper:
    def __init__(self, is_train=True):
        self.is_train = is_train
        sz = list(cfg.INPUT.MIN_SIZE_TRAIN) if is_train else [cfg.INPUT.MIN_SIZE_TEST]
        mx = cfg.INPUT.MAX_SIZE_TRAIN if is_train else cfg.INPUT.MAX_SIZE_TEST
        self.augs = T.AugmentationList([T.ResizeShortestEdge(sz, mx, sample_style='choice')])

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        img = imreader._read_image(dataset_dict['file_name'])
        aug_input  = T.AugInput(img)
        transforms = self.augs(aug_input)
        img = aug_input.image
        dataset_dict['image'] = torch.as_tensor(np.ascontiguousarray(img.transpose(2, 0, 1)))
        if not self.is_train:
            dataset_dict.pop('annotations', None)
            return dataset_dict
        annos = [utils.transform_instance_annotations(obj, transforms, img.shape[:2])
                 for obj in dataset_dict.pop('annotations', []) if obj.get('iscrowd', 0) == 0]
        dataset_dict['instances'] = utils.annotations_to_instances(annos, img.shape[:2])
        return dataset_dict

class PlateTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=PNGMapper(is_train=True))
    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        return build_detection_test_loader(cfg, dataset_name, mapper=PNGMapper(is_train=False))
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, 'eval')
        return COCOEvaluator(dataset_name, output_dir=output_folder)

trainer = PlateTrainer(cfg)
trainer.resume_or_load(resume=True)
print(f'Training: {cfg.SOLVER.MAX_ITER} iterations  |  checkpoints every {cfg.SOLVER.CHECKPOINT_PERIOD}')
print('-' * 60)
trainer.train()
print('-' * 60)
print(f'Training complete. Model: {MODEL_DIR}/model_final.pth')

# ── Loss plot ─────────────────────────────────────────────────────────────────
metrics_path = MODEL_DIR / 'metrics.json'
if metrics_path.exists():
    records    = [json.loads(l) for l in metrics_path.read_text().splitlines() if l.strip()]
    iters      = [r['iteration']        for r in records if 'total_loss' in r]
    total_loss = [r['total_loss']       for r in records if 'total_loss' in r]
    fig, axes  = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(iters, total_loss, color='white', lw=1.5, label='total_loss')
    for vals, label, color in [
        ([r.get('loss_cls')     for r in records if 'total_loss' in r], 'loss_cls',     'cyan'),
        ([r.get('loss_box_reg') for r in records if 'total_loss' in r], 'loss_box_reg', 'orange'),
        ([r.get('loss_rpn_cls') for r in records if 'total_loss' in r], 'loss_rpn_cls', 'magenta'),
        ([r.get('loss_rpn_loc') for r in records if 'total_loss' in r], 'loss_rpn_loc', 'yellow'),
    ]:
        if any(v is not None for v in vals):
            axes[1].plot(iters, vals, color=color, lw=1.2, label=label)
    for ax, title in [(axes[0], 'Total Loss'), (axes[1], 'Loss Components')]:
        ax.set_facecolor('#111'); ax.set_title(title, color='white')
        ax.set_xlabel('Iteration', color='white'); ax.tick_params(colors='white')
        for sp in ax.spines.values(): sp.set_edgecolor('#444')
    axes[0].legend(facecolor='#222', labelcolor='white')
    axes[1].legend(facecolor='#222', labelcolor='white')
    fig.patch.set_facecolor('#1a1a1a')
    plt.suptitle(f'Training losses — {len(iters)} iterations logged', color='white', fontsize=11)
    plt.tight_layout(); plt.show()
    print(f'Final total_loss: {total_loss[-1]:.4f}  (iteration {iters[-1]})')

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5: DEEPDISC INFERENCE + PHOTOMETRY
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('SECTION 5: DeepDISC inference + photometry')
print('='*60)

if not (MODEL_DIR / 'model_final.pth').exists():
    print('ERROR: model_final.pth not found. Training may have failed.')
else:
    cfg_infer = cfg.clone()
    cfg_infer.MODEL.WEIGHTS                     = str(MODEL_DIR / 'model_final.pth')
    cfg_infer.MODEL.DEVICE                      = 'cpu'
    cfg_infer.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.4
    predictor = DefaultPredictor(cfg_infer)
    INFER_VER = 1
    print(f'Model loaded  |  score threshold: {cfg_infer.MODEL.ROI_HEADS.SCORE_THRESH_TEST}')

    overall_bar   = widgets.IntProgress(value=0, min=0, max=len(star_dirs), description='Stars:')
    overall_label = widgets.Label('Starting...')
    plate_bar     = widgets.IntProgress(value=0, min=0, max=1, description='Plates:')
    plate_html    = widgets.HTML()
    display(widgets.VBox([widgets.HBox([overall_bar, overall_label]), plate_bar, plate_html]))

    run_summaries = []

    for si, star_dir in enumerate(star_dirs):
        overall_label.value = f'{star_dir.name} ({si+1}/{len(star_dirs)})'
        target_coord, _ = target_coords[star_dir.name]
        if target_coord is None:
            print(f'  ⚠ {star_dir.name}: no coordinate — skipping')
            overall_bar.value = si + 1; continue

        cutouts  = sorted((star_dir / 'cutouts').glob('*.fits'))
        out_dir  = star_dir / '03_B'; out_dir.mkdir(parents=True, exist_ok=True)
        ic_dir   = out_dir / 'infer_cache'; ic_dir.mkdir(parents=True, exist_ok=True)
        infer_db = shelve.open(str(ic_dir / 'infer_shelf'), flag='c', writeback=False)
        total    = len(cutouts)
        plate_bar.max = total; plate_bar.value = 0
        start = time.monotonic(); last_render = 0.0
        result_rows = []

        for pi, fits_path in enumerate(cutouts):
            mtime  = fits_path.stat().st_mtime
            key    = str(fits_path)
            cached = infer_db.get(key)

            if (cached is not None
                    and abs(cached.get('mtime', 0) - mtime) < 1.0
                    and cached.get('_ver', 0) == INFER_VER):
                result_rows.append(cached['result_row'])
            else:
                rr = {'filename': fits_path.name, 'jd': np.nan,
                       'target_detected': False, 'target_mag': np.nan,
                       'target_mag_error': np.nan, 'num_reference_stars': 0,
                       'rms_scatter': np.nan, 'calibration_mode': None,
                       'quality_ok': False, 'rejection_reason': 'not processed'}
                try:
                    data   = astrofits.getdata(fits_path).astype(float)
                    header = astrofits.getheader(fits_path)
                    wcs    = WCS(header)
                    rr['jd'] = get_plate_jd(header)

                    png_path = IMG_DIR / (fits_path.stem + '.png')
                    if not png_path.exists():
                        fits_to_png(fits_path, png_path)
                    img_bgr = cv2.imread(str(png_path))
                    if img_bgr is None:
                        img_rgb = np.array(PILImage.open(png_path).convert('RGB'))
                        img_bgr = img_rgb[:, :, ::-1].copy()

                    out       = predictor(img_bgr)
                    instances = out['instances'].to('cpu')
                    boxes     = instances.pred_boxes.tensor.numpy()
                    pred_cats = instances.pred_classes.numpy()
                    smask     = (pred_cats == 0) | (pred_cats == 1)
                    if smask.sum() == 0:
                        rr['rejection_reason'] = 'No sources detected'; raise StopIteration
                    src_x = (boxes[smask, 0] + boxes[smask, 2]) / 2
                    src_y = (boxes[smask, 1] + boxes[smask, 3]) / 2

                    h, w = data.shape
                    ra_c, dec_c = wcs.all_pix2world(w/2, h/2, 0)
                    cra, cdec   = wcs.all_pix2world([0,w-1,0,w-1],[0,0,h-1,h-1],0)
                    ctr = SkyCoord(float(ra_c)*u.deg, float(dec_c)*u.deg)
                    fr  = ctr.separation(SkyCoord(cra*u.deg, cdec*u.deg)).max().arcsec * 1.05
                    apass_stars = get_field_catalog(float(ra_c), float(dec_c), fr)
                    if len(apass_stars) < MIN_APASS_STARS_QUERY:
                        rr['rejection_reason'] = 'Too few APASS stars'; raise StopIteration

                    ara = np.array([s[0] for s in apass_stars])
                    adc = np.array([s[1] for s in apass_stars])
                    ab  = np.array([s[2] for s in apass_stars])
                    ax, ay = wcs.all_world2pix(ara, adc, 0)
                    mg = 20
                    inf = (ax>mg)&(ax<w-mg)&(ay>mg)&(ay<h-mg)
                    ax, ay, ab = ax[inf], ay[inf], ab[inf]
                    if len(ax) == 0:
                        rr['rejection_reason'] = 'No APASS stars in frame'; raise StopIteration

                    data_sub, _ = subtract_background_2d(data)
                    axr, ayr    = refine_centroids(data_sub, ax, ay)
                    iso         = filter_isolated(axr, ayr)
                    fl, _, bs   = measure_photometry(data_sub, axr, ayr)
                    area        = np.pi * DEFAULT_APERTURE**2
                    with np.errstate(divide='ignore', invalid='ignore'):
                        snr = np.where(bs > 0, fl / (bs * np.sqrt(area)), 0)
                        im  = np.where(fl > 0, -2.5 * np.log10(np.maximum(fl, 1e-10)), np.nan)
                    sat  = detect_saturation(data, axr, ayr)
                    good = (snr >= SIGNIF_THRESHOLD) & np.isfinite(im) & ~sat & iso

                    cal_g = calibrate_global(im[good], ab[good])
                    if cal_g is None:
                        rr['rejection_reason'] = 'Global calibration failed'; raise StopIteration
                    rr['num_reference_stars'] = cal_g['n_used']

                    tx, ty = wcs.all_world2pix(target_coord.ra.deg, target_coord.dec.deg, 0)
                    if not (mg < tx < w-mg and mg < ty < h-mg):
                        rr['rejection_reason'] = 'Target outside frame'; raise StopIteration
                    dists = np.hypot(src_x - tx, src_y - ty)
                    if dists.min() > 20:
                        rr['rejection_reason'] = 'No detection near target'; raise StopIteration

                    ni = int(np.argmin(dists))
                    txr, tyr = refine_centroids(data_sub, np.array([src_x[ni]]), np.array([src_y[ni]]))
                    tf, _, tbs = measure_photometry(data_sub, txr, tyr)
                    tsnr = tf[0] / (tbs[0] * np.sqrt(area)) if tbs[0] > 0 else 0
                    if tsnr < SIGNIF_THRESHOLD or tf[0] <= 0:
                        rr['rejection_reason'] = 'Target SNR too low'; raise StopIteration

                    ti    = -2.5 * np.log10(tf[0])
                    rough = invert_calibration(cal_g, ti)
                    if rough is None:
                        rr['rejection_reason'] = 'Calibration inversion failed'; raise StopIteration

                    cal_l = calibrate_local(im[good], ab[good], rough)
                    if cal_l is not None:
                        lm = invert_calibration(cal_l, ti)
                        if lm is not None:
                            tmag = lm; frms = cal_l['rms']; fn = cal_l['n_used']; cmode = 'local'
                        else:
                            tmag = rough; frms = cal_g['rms']; fn = cal_g['n_used']; cmode = 'global_fallback'
                    else:
                        tmag = rough; frms = cal_g['rms']; fn = cal_g['n_used']; cmode = 'global_fallback'

                    terr = np.sqrt((1.0857 / max(tsnr, 1e-6))**2 + (frms / np.sqrt(max(fn, 1)))**2)
                    qok  = fn >= MIN_REF_STARS and np.isfinite(frms) and frms <= MAX_CALIBRATION_RMS
                    rr.update({'target_detected': True, 'target_mag': tmag,
                               'target_mag_error': terr, 'rms_scatter': frms,
                               'calibration_mode': cmode, 'quality_ok': qok,
                               'rejection_reason': None if qok else f'RMS {frms:.3f}'})
                except StopIteration:
                    pass
                except Exception as e:
                    rr['rejection_reason'] = str(e)

                infer_db[key] = {'result_row': rr, 'mtime': mtime, '_ver': INFER_VER}
                result_rows.append(rr)

            now = time.monotonic()
            if now - last_render > 0.15 or (pi + 1) == total:
                el = now - start; rate = (pi+1)/el if el > 0 else 0
                rem = (total-pi-1)/rate if rate > 0 else None
                plate_bar.value = pi + 1
                plate_html.value = (f"<div style='font-family:monospace;font-size:12px'>"
                                    f"<b>{(pi+1)/total*100:5.1f}%</b> &nbsp; {pi+1:,}/{total:,} &nbsp;|&nbsp; "
                                    f"{rate:.2f} pl/s &nbsp;|&nbsp; Elapsed {format_eta(el)} &nbsp;|&nbsp; ETA {format_eta(rem)}"
                                    f"</div>")
                last_render = now

        infer_db.sync(); infer_db.close()
        df = pd.DataFrame(result_rows).sort_values('jd').reset_index(drop=True)
        df.to_csv(out_dir / 'photometry_results.csv', index=False)

        lc = df[df['quality_ok']].copy()
        lc = lc.rename(columns={'target_mag': 'magnitude', 'target_mag_error': 'magnitude_error'})
        lc = lc[['jd', 'magnitude', 'magnitude_error']].dropna().sort_values('jd').reset_index(drop=True)
        n_before = len(lc)
        if len(lc) > 20:
            rm = lc['magnitude'].rolling(15, center=True, min_periods=5).median()
            cl = sigma_clip(lc['magnitude'] - rm, sigma=4, maxiters=3)
            lc = lc[~cl.mask].reset_index(drop=True)
        lc.to_csv(out_dir / 'lightcurve.csv', index=False)

        run_summaries.append({'star': star_dir.name, 'n_plates': len(df),
                               'n_lc_points': len(lc), 'n_clipped': n_before - len(lc)})
        print(f'  ✓ {star_dir.name}: {len(df)} plates, {len(lc)} light curve points')
        apass_cache.sync()
        overall_bar.value = si + 1

    overall_label.value = f'Done. {len(run_summaries)}/{len(star_dirs)} stars.'
    apass_cache.sync()

    print('\n' + '='*60)
    print('RUN SUMMARY')
    print('='*60)
    if run_summaries:
        print(pd.DataFrame(run_summaries).to_string(index=False))
    print('\nRun Cell 2 to view light curves and compare with DASCH.')


c:\Users\dapur\miniconda3\envs\deepdisc_IG\Lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Found 14 star folder(s):
  R_CrB: 10555 plates
  RS_Tel: 5265 plates
  RY_Sgr: 7003 plates
  S_Aps: 6820 plates
  SU_Tau: 10272 plates
  U_Aqr: 8149 plates
  UW_Cen: 4771 plates
  V2552_Oph: 6806 plates
  V348_Sgr: 7084 plates
  V581_CrA: 5183 plates
  V854_Cen: 4542 plates
  V_CrA: 5405 plates
  XX_Cam: 10643 plates
  Y_Mus: 5760 plates

APASS cache: 0 field queries
  R_CrB: OK (resolved)
  RS_Tel: OK (resolved)
  RY_Sgr: OK (resolved)
  S_Aps: OK (resolved)
  SU_Tau: OK (resolved)
  U_Aqr: OK (resolved)
  UW_Cen: OK (resolved)
  V2552_Oph: OK (resolved)
  V348_Sgr: OK (resolved)
  V581_CrA: OK (resolved)
  V854_Cen: OK (resolved)
  V_CrA: OK (manual)
  XX_Cam: OK (resolved)
  Y_Mus: OK (resolved)

SECTION 1: Building COCO dataset
No existing cache -- full build


  R_CrB: 9341 new | 0 cached | 1214 skipped


  RS_Tel: 4857 new | 0 cached | 408 skipped


  RY_Sgr: 5945 new | 0 cached | 1058 skipped


...

  V348_Sgr: 5647 new | 0 cached | 1437 skipped


  V581_CrA: 4700 new | 0 cached | 483 skipped


  V854_Cen: 4047 new | 0 cached | 495 skipped


  V_CrA: 4712 new | 0 cached | 693 skipped


XX_Cam:  55%|█████▍    | 5811/10643 [2:47:06<1:57:01,  1.45s/it] 

In [ ]:
# NOTE: 03_B_DeepDisc.ipynb - Cell 2
# Light curve comparison: DeepDISC pipeline vs DASCH.
# Self-contained -- can be re-run any time after Cell 1 has saved output files,
# even after a kernel restart. Identical plot style to 02_A Cell 2.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DATA_DIR = Path(r'C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\data')

star_dirs = sorted([d for d in BASE_DATA_DIR.iterdir()
                     if d.is_dir() and (d / 'cutouts').is_dir()
                     and not d.name.startswith('_')])

plot_summaries = []

for star_dir in star_dirs:
    lc_path = star_dir / '03_B' / 'lightcurve.csv'
    if not lc_path.exists():
        print(f'⚠ {star_dir.name}: no lightcurve.csv found -- run Cell 1 first.')
        continue

    lc_df = pd.read_csv(lc_path)

    dasch_jd = dasch_mag = dasch_err = None
    try:
        from daschlab import Session
        sess = Session(str(star_dir))
        sess.select_target(star_dir.name.replace('_', ' '))
        sess.select_refcat('apass')
        dlc  = sess.lightcurve(0)
        dj   = np.asarray(dlc['time'].jd)
        dm   = np.asarray(dlc['magcal_magdep'].value)
        de   = (np.asarray(dlc['magcal_local_rms'].value)
                if 'magcal_local_rms' in dlc.colnames
                else np.full_like(dm, np.nan))
        good = np.isfinite(dj) & np.isfinite(dm)
        dasch_jd, dasch_mag, dasch_err = dj[good], dm[good], de[good]
    except Exception as e:
        print(f'  ⚠ DASCH unavailable for {star_dir.name}: {e}')

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

    if len(lc_df) > 0:
        ax1.errorbar(lc_df['jd'], lc_df['magnitude'], yerr=lc_df['magnitude_error'],
                     fmt='o', color='#3498db', markersize=4,
                     ecolor='#ef00ff', elinewidth=1.0, capsize=1.5, capthick=0.6,
                     alpha=0.9, markeredgewidth=0)
        ax1.invert_yaxis()
    else:
        ax1.text(0.5, 0.5, 'No quality-passing detections',
                 ha='center', va='center', transform=ax1.transAxes)
    ax1.set_ylabel('B Magnitude')
    ax1.set_title(f'{star_dir.name} — DeepDISC Pipeline ({len(lc_df)} points)')
    ax1.grid(True, alpha=0.3)

    if dasch_jd is not None and len(dasch_jd) > 0:
        ax2.errorbar(dasch_jd, dasch_mag, yerr=dasch_err,
                     fmt='o', color='black', markersize=3,
                     ecolor='#e74c3c', elinewidth=0.6, capsize=1.2, capthick=0.5,
                     alpha=0.85, markeredgewidth=0)
        ax2.invert_yaxis()
        ax2.set_title(f'{star_dir.name} — DASCH Reference ({len(dasch_jd)} points)')
    else:
        ax2.text(0.5, 0.5, 'DASCH light curve unavailable',
                 ha='center', va='center', transform=ax2.transAxes)
        ax2.set_title(f'{star_dir.name} — DASCH Reference (unavailable)')
    ax2.set_xlabel('Julian Date')
    ax2.set_ylabel('Calibrated Magnitude')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    plot_summaries.append({
        'star':              star_dir.name,
        'n_deepdisc_points': len(lc_df),
        'n_dasch_points':    len(dasch_jd) if dasch_jd is not None else 0,
    })

print('\n' + '='*60)
print('PLOT SUMMARY')
print('='*60)
if plot_summaries:
    print(pd.DataFrame(plot_summaries).to_string(index=False))
